# 3. Portfolio Construction and Risk Adjustment

## Overview
This notebook implements the portfolio construction process across five asset classes with dynamic, risk-adjusted weight allocation:

1. **Strategic Asset Class Allocation**
   - Equities: 65% | Bonds: 10% | Precious Metals: 5% | Energy: 5% | Agriculture: 5%
   - Regional diversification within equities and bonds

2. **Sharpe Ratio Adjustments**
   - Weights adjusted up or down based on risk-adjusted returns vs. each asset class benchmark
   - Per-class sensitivity controls how much the Sharpe ratio can shift weights:
     - Equities: ±0.1 | Bonds: ±0.25 | Precious Metals: ±0.15 | Energy: ±0.15 | Agriculture: ±0.15

3. **Final Portfolio Construction**
   - Risk-adjusted position sizing across all five asset classes
   - Volatility-based cash weight adjustments
   - Final investment allocation in GBP

4. **Portfolio Versioning**
   - 2025 portfolio (equity + bonds only) is seeded from `final_portfolio_25.csv` and **locked** on first run
   - 2026 portfolio (all five asset classes) is saved to the `portfolios` table and remains **mutable** until explicitly locked

The process aims to create a well-diversified portfolio that balances return potential with risk management.

## Setup and Data Loading

In [ ]:
from pathlib import Path
import sys
sys.path.insert(0, str(Path("..").resolve()))

import pandas as pd
import numpy as np

from etf_utils.config import DATA_RAW, DATA_INTERMEDIATE, DATA_OUTPUT, DATA_CONFIG
from etf_utils.data_provider import DataProvider
from etf_utils.data_io import load_intermediate, save_output
from etf_utils.metrics import calculate_annualized_volatility, interpolate_adjustment_factor
from etf_utils.database import load_screened_etfs, save_portfolio, seed_2025_portfolio

provider = DataProvider()

# Seed the locked 2025 portfolio (no-op if already done)
seed_2025_portfolio()

# Load screened ETFs for 2026 from the DB (falls back to summary_all.csv if DB is empty)
etfs_df = load_screened_etfs(portfolio_year=2026)
if etfs_df.empty:
    etfs_df = pd.read_csv(DATA_INTERMEDIATE / 'summary_all.csv')
print(f"Loaded {len(etfs_df)} ETFs for portfolio construction (asset classes: {etfs_df['asset_class'].unique().tolist()})")

## Risk Adjustment Framework

### Sharpe Ratio Adjustment System
We implement a dynamic weight adjustment system based on Sharpe ratios relative to each asset class benchmark:

1. **Base Factors**: Scale from 0.6 (poor) to 1.48 (excellent)
2. **Relative Performance**: Adjustments based on deviation from the intra-class median Sharpe ratio
3. **Asset Class Sensitivity** (maximum shift in adjustment factor):
   - Equities: ±0.1
   - Bonds: ±0.25
   - Precious Metals: ±0.15
   - Energy: ±0.15
   - Agriculture: ±0.15

This allows the portfolio to:
- Reward strong risk-adjusted performance
- Reduce exposure to high-risk/low-return assets
- Apply appropriate sensitivity by asset class

In [ ]:
# Sharpe ratio adjustment factors (top-level, across all asset classes)
trailing_sr_adjustment_factors = {
    -1.0: 0.60,
    -0.8: 0.66,
    -0.6: 0.77,
    -0.4: 0.85,
    -0.2: 0.94,
     0.0: 1.00,
     0.2: 1.11,
     0.4: 1.19,
     0.6: 1.30,
     0.8: 1.37,
     1.0: 1.48
}

# Initial risk weights — 4 asset classes: 65 / 10 / 5 / 10
eq_risk_weight  = 65   # Equities
bnd_risk_weight = 10   # Bonds
pm_risk_weight  = 5    # Precious Metals
cmd_risk_weight = 10   # Commodities (broad: energy ~30%, agri ~20%, metals)

# Regional risk weights for equity/bonds (unchanged)
regional_risk_weights_data = {
    'category': [
        'Developed_AmericasandUK',
        'Developed_EMEA',
        'Developed_APAC',
        'Emerging_Americas',
        'Emerging_APACandEMEA'
    ],
    'allocation': [10, 35, 35, 10, 10],
}
regional_risk_weights = pd.DataFrame(regional_risk_weights_data)

# Intra-asset Sharpe adjustment tables
# Equities: ±0.1 sensitivity
intra_asset_equity_trailing_sr_data = {
    -0.1: 0.6, -0.08: 0.66, -0.06: 0.77, -0.04: 0.85, -0.02: 0.94,
     0:   1.0,  0.02: 1.11,  0.04: 1.19,  0.06: 1.30,  0.08: 1.37, 0.1: 1.48
}

# Bonds: ±0.25 sensitivity
intra_asset_bond_trailing_sr_data = {
    -0.25: 0.6, -0.2: 0.66, -0.15: 0.77, -0.1: 0.85, -0.05: 0.94,
     0:    1.0,  0.05: 1.11,  0.1: 1.19,  0.15: 1.30,  0.2: 1.37, 0.25: 1.48
}

# Precious Metals: ±0.15 sensitivity
intra_asset_pm_trailing_sr_data = {
    -0.15: 0.6, -0.12: 0.66, -0.09: 0.77, -0.06: 0.85, -0.03: 0.94,
     0:    1.0,  0.03: 1.11,  0.06: 1.19,  0.09: 1.30,  0.12: 1.37, 0.15: 1.48
}

# Commodities: ±0.15 sensitivity
intra_asset_cmd_trailing_sr_data = {
    -0.15: 0.6, -0.12: 0.66, -0.09: 0.77, -0.06: 0.85, -0.03: 0.94,
     0:    1.0,  0.03: 1.11,  0.06: 1.19,  0.09: 1.30,  0.12: 1.37, 0.15: 1.48
}

# Map asset class → intra-asset SR table
sr_data_map = {
    'equity':         intra_asset_equity_trailing_sr_data,
    'bonds':          intra_asset_bond_trailing_sr_data,
    'preciousMetals': intra_asset_pm_trailing_sr_data,
    'commodities':    intra_asset_cmd_trailing_sr_data,
}


## Performance Data Collection

The following code block retrieves and calculates key performance metrics for all four benchmark ETFs:

| Asset Class | Benchmark | Ticker |
|---|---|---|
| Equities | Vanguard FTSE Dev World | VEVE.L |
| Bonds | SPDR Bloomberg 0–3Y US Agg | SAAA.L |
| Precious Metals | iShares Physical Gold ETC | SGLN.L |
| Commodities | Invesco Bloomberg Commodity | CMOP.L |

Metrics collected for 2025: total return, annualised volatility, and Sharpe ratio. These form the baseline for Sharpe ratio adjustment calculations.

In [ ]:
# Calculate benchmark returns and volatility for 2025 — all 4 asset classes
benchmark_year_start = "2025-01-01"
benchmark_year_end   = "2025-12-31"

def _benchmark_metrics(symbol, start, end):
    prices = provider.get_historical_prices(symbol, start_date=start, end_date=end)
    yr = prices["close"]
    ret = round((yr.iloc[-1] - yr.iloc[0]) / yr.iloc[0] * 100, 2)
    vol = round(calculate_annualized_volatility(yr) * 100, 2)
    sr  = round(ret / vol, 2)
    return ret, vol, sr

eq_benchmark_return,  eq_benchmark_volatility,  eq_sharpe_ratio  = _benchmark_metrics("VEVE", benchmark_year_start, benchmark_year_end)
bnd_benchmark_return, bnd_benchmark_volatility, bond_sharpe_ratio = _benchmark_metrics("SAAA", benchmark_year_start, benchmark_year_end)
pm_benchmark_return,  pm_benchmark_volatility,  pm_sharpe_ratio  = _benchmark_metrics("SGLN", benchmark_year_start, benchmark_year_end)
cmd_benchmark_return, cmd_benchmark_volatility, cmd_sharpe_ratio = _benchmark_metrics("CMOP", benchmark_year_start, benchmark_year_end)

print(f"2025 VEVE  return: {eq_benchmark_return}%,  vol: {eq_benchmark_volatility}%,  Sharpe: {eq_sharpe_ratio}")
print(f"2025 SAAA  return: {bnd_benchmark_return}%, vol: {bnd_benchmark_volatility}%, Sharpe: {bond_sharpe_ratio}")
print(f"2025 SGLN  return: {pm_benchmark_return}%,  vol: {pm_benchmark_volatility}%,  Sharpe: {pm_sharpe_ratio}")
print(f"2025 CMOP  return: {cmd_benchmark_return}%, vol: {cmd_benchmark_volatility}%, Sharpe: {cmd_sharpe_ratio}")

## Weight Interpolation Function

This helper function implements the core logic for Sharpe ratio adjustments:

1. Takes a Sharpe ratio and the asset class sensitivity table as input
2. Maps the ratio to the appropriate adjustment factor via linear interpolation
3. Handles edge cases (above/below table range) by clamping to boundary values
4. Returns the interpolated weight adjustment factor

The same function is reused for all five asset classes; only the sensitivity table (±0.1 / ±0.25 / ±0.15) differs.

In [ ]:
# interpolate_adjustment_factor is imported from etf_utils.metrics
# Example usage:
# factor = interpolate_adjustment_factor(sharpe_ratio, trailing_sr_adjustment_factors)

## Final Portfolio Construction

The following calculations implement the complete portfolio construction process:

1. **Intra-class Risk Weight Calculation**
   - Compute median Sharpe ratio per asset class
   - Apply `interpolate_adjustment_factor` to each ETF relative to its class median
   - Normalise adjusted weights within each asset class

2. **Cross-class Allocation**
   - Apply strategic class weights: 65 / 10 / 5 / 5 / 5
   - Combine into a single ranked portfolio DataFrame

3. **Cash Weight Determination**
   - Convert risk weights to cash weights adjusted for volatility

4. **Final Allocation**
   - Calculate GBP investment amounts
   - Ensure diversification across all five asset classes
   - Save to `portfolios` table (year = 2026) and `final_portfolio.csv`

In [ ]:
# ── Top-level asset-class weight normalisation (4 classes) ──────────────────────
eq_adj_factor  = interpolate_adjustment_factor(eq_sharpe_ratio,  trailing_sr_adjustment_factors)
bond_adj_factor = interpolate_adjustment_factor(bond_sharpe_ratio, trailing_sr_adjustment_factors)
pm_adj_factor  = interpolate_adjustment_factor(pm_sharpe_ratio,  trailing_sr_adjustment_factors)
cmd_adj_factor = interpolate_adjustment_factor(cmd_sharpe_ratio, trailing_sr_adjustment_factors)

eq_adj  = eq_risk_weight  * eq_adj_factor
bnd_adj = bnd_risk_weight * bond_adj_factor
pm_adj  = pm_risk_weight  * pm_adj_factor
cmd_adj = cmd_risk_weight * cmd_adj_factor

total_adj = eq_adj + bnd_adj + pm_adj + cmd_adj
normalized_eq_weight   = round(eq_adj  / total_adj * 100, 2)
normalized_bond_weight = round(bnd_adj / total_adj * 100, 2)
normalized_pm_weight   = round(pm_adj  / total_adj * 100, 2)
normalized_cmd_weight  = round(cmd_adj / total_adj * 100, 2)
total_normalized_weight = normalized_eq_weight + normalized_bond_weight + normalized_pm_weight + normalized_cmd_weight

# Volatility-adjusted cash weights per asset class
eq_cash_weights  = normalized_eq_weight  / eq_benchmark_volatility
bond_cash_weights = normalized_bond_weight / bnd_benchmark_volatility
pm_cash_weights  = normalized_pm_weight  / pm_benchmark_volatility
cmd_cash_weights = normalized_cmd_weight / cmd_benchmark_volatility

total_cash_weight     = eq_cash_weights + bond_cash_weights + pm_cash_weights + cmd_cash_weights
final_eq_cash_weight   = total_normalized_weight * eq_cash_weights  / total_cash_weight
final_bond_cash_weight = total_normalized_weight * bond_cash_weights / total_cash_weight
final_pm_cash_weight   = total_normalized_weight * pm_cash_weights  / total_cash_weight
final_cmd_cash_weight  = total_normalized_weight * cmd_cash_weights / total_cash_weight

print("Asset Class Cash Weights (volatility-adjusted):")
print(f"  Equities:        {final_eq_cash_weight:.2f}%")
print(f"  Bonds:           {final_bond_cash_weight:.2f}%")
print(f"  Precious Metals: {final_pm_cash_weight:.2f}%")
print(f"  Commodities:     {final_cmd_cash_weight:.2f}%")
total_check = final_eq_cash_weight + final_bond_cash_weight + final_pm_cash_weight + final_cmd_cash_weight
print(f"  Total:           {total_check:.2f}%")

In [ ]:
etfs = etfs_df.copy()

# ── Region category weight ──────────────────────────────────────────────────────
# Precious Metals and Commodities use a single Global region → 100% weight
_regional_lookup = regional_risk_weights.set_index('category')['allocation']

def _get_region_weight(row):
    if row['region_category'] == 'Global':
        return 100
    return _regional_lookup.get(row['region_category'], 0)

etfs['region_category_weight'] = etfs.apply(_get_region_weight, axis=1)

# ── Intra-region category weight ───────────────────────────────────────────────
etfs['intra_region_category_weight'] = 0

for asset in etfs['asset_class'].unique():
    for region_category in etfs[etfs['asset_class'] == asset]['region_category'].unique():
        mask = (etfs['asset_class'] == asset) & (etfs['region_category'] == region_category)
        num_categories = len(etfs[mask]['intra_region_category'].unique())
        if num_categories > 0:
            etfs.loc[mask, 'intra_region_category_weight'] = 100 // num_categories

# ── Sharpe ratio calculations ──────────────────────────────────────────────────
etfs['starting_risk_weights'] = etfs['region_category_weight'] * etfs['intra_region_category_weight'] / 100
etfs['yield'] = (etfs['last_year_dividends'] - etfs['ter']).round(2)
etfs['sharpe_ratio'] = (etfs['yield'] / etfs['last_year_volatility']).round(2)

# Median Sharpe per asset class
median_sharpe = {
    ac: etfs[etfs['asset_class'] == ac]['sharpe_ratio'].median()
    for ac in etfs['asset_class'].unique()
}

etfs['relative_sharpe_ratio'] = etfs.apply(
    lambda row: row['sharpe_ratio'] - median_sharpe[row['asset_class']], axis=1
)
etfs['interpolated_adjusted_factors'] = etfs.apply(
    lambda row: interpolate_adjustment_factor(
        row['relative_sharpe_ratio'], sr_data_map[row['asset_class']]
    ), axis=1
)

# ── Risk weights ───────────────────────────────────────────────────────────────
etfs['adjusted_risk_weights'] = (
    etfs['interpolated_adjusted_factors'] * etfs['starting_risk_weights']
).round(2)

# Normalize within each asset class to sum to 100%
etfs['normalized_risk_weights'] = etfs.groupby('asset_class')['adjusted_risk_weights'].transform(
    lambda x: x / x.sum() * 100
)

# ── Cash weights (intra-asset class) ──────────────────────────────────────────
etfs['cash_weights_guess'] = etfs['normalized_risk_weights'] / etfs['last_year_volatility']

etfs['cash_weights'] = etfs.groupby('asset_class')['cash_weights_guess'].transform(
    lambda x: (etfs.loc[x.index, 'normalized_risk_weights'].sum() / x.sum()) * x
).round(2)

# Final regional category weight (sum of cash weights per region per asset class)
etfs['region_category_weight_final'] = etfs.groupby(
    ['asset_class', 'region_category']
)['cash_weights'].transform('sum')

# ── Asset class weight map (4 classes) ────────────────────────────────────────
asset_class_weight_map = {
    'equity':         normalized_eq_weight,
    'bonds':          normalized_bond_weight,
    'preciousMetals': normalized_pm_weight,
    'commodities':    normalized_cmd_weight,
}
etfs['normalized_asset_class_weight'] = etfs['asset_class'].map(asset_class_weight_map)

# ── Final weights (all 4 asset classes together) ──────────────────────────────
etfs['final_risk_weights'] = etfs['normalized_asset_class_weight'] * etfs['normalized_risk_weights'] / 100
etfs['final_cash_weights_guess'] = etfs['final_risk_weights'] / etfs['last_year_volatility']
etfs['final_cash_weights'] = (
    etfs['final_risk_weights'].sum() / etfs['final_cash_weights_guess'].sum()
    * etfs['final_cash_weights_guess']
).round()

# ── Investment amounts (GBP) ───────────────────────────────────────────────────
etfs['investment'] = 20000 * etfs['final_cash_weights'] / 100

# ── Save to DB (year=2026, overwritable) and CSV backup ──────────────────────
save_portfolio(etfs, year=2026)
etfs.to_csv(DATA_OUTPUT / 'final_portfolio.csv', index=False)

etfs.sort_values(by=['asset_class', 'investment'], ascending=[False, False])[[
    'asset_class', 'region_category', 'country', 'intra_region_category',
    'ticker', 'name', 'final_cash_weights', 'beta', 'yday_close_price',
    'ter', 'investment', 'available_on_investengine'
]]
